# 01 — Create Sample Order Batches

**Project:** Incremental Orders Pipeline with Delta Lake  
**Layer:** Data Generation  
**Responsibility:** Create sample incremental order batches to simulate new orders and order status updates.

This notebook creates synthetic order event batches that will be used to practice incremental ingestion, upserts, deduplication, and Delta Lake `MERGE INTO`.

The data simulates an e-commerce system where orders can change status over time:

- `pending`
- `shipped`
- `delivered`
- `cancelled`

The goal of the project is to maintain a reliable current-state Silver table using incremental data processing.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS orders_project")
spark.sql("USE SCHEMA orders_project")

In [0]:
from datetime import datetime
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType, IntegerType
)

In [0]:
order_event_schema = StructType([
    StructField("batch_id", IntegerType(), False),
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), False),
    StructField("order_status", StringType(), False),
    StructField("order_amount", DoubleType(), False),
    StructField("event_ts", TimestampType(), False),
    StructField("source_system", StringType(), False)
])

In [0]:
batch_1_data = [
    (1, "1001", "C001", "pending", 250.00, datetime(2026, 6, 1, 10, 0, 0), "ecommerce_orders"),
    (1, "1002", "C002", "pending", 125.50, datetime(2026, 6, 1, 10, 5, 0), "ecommerce_orders"),
    (1, "1003", "C003", "pending", 780.90, datetime(2026, 6, 1, 10, 10, 0), "ecommerce_orders")
]

batch_1_df = spark.createDataFrame(batch_1_data, order_event_schema)

display(batch_1_df)

In [0]:
batch_2_data = [
    (2, "1001", "C001", "shipped", 250.00, datetime(2026, 6, 1, 12, 0, 0), "ecommerce_orders"),
    (2, "1002", "C002", "cancelled", 125.50, datetime(2026, 6, 1, 12, 15, 0), "ecommerce_orders"),
    (2, "1004", "C004", "pending", 310.75, datetime(2026, 6, 1, 12, 30, 0), "ecommerce_orders")
]

batch_2_df = spark.createDataFrame(batch_2_data, order_event_schema)

display(batch_2_df)

In [0]:
batch_3_data = [
    (3, "1001", "C001", "delivered", 250.00, datetime(2026, 6, 2, 9, 0, 0), "ecommerce_orders"),
    (3, "1003", "C003", "shipped", 780.90, datetime(2026, 6, 2, 9, 15, 0), "ecommerce_orders"),
    (3, "1004", "C004", "delivered", 310.75, datetime(2026, 6, 2, 9, 30, 0), "ecommerce_orders"),
    (3, "1005", "C005", "pending", 95.25, datetime(2026, 6, 2, 9, 45, 0), "ecommerce_orders")
]

batch_3_df = spark.createDataFrame(batch_3_data, order_event_schema)

display(batch_3_df)

In [0]:
batch_1_df.createOrReplaceTempView("batch_1_orders")
batch_2_df.createOrReplaceTempView("batch_2_orders")
batch_3_df.createOrReplaceTempView("batch_3_orders")

print("Temporary views created:")
print("- batch_1_orders")
print("- batch_2_orders")
print("- batch_3_orders")

In [0]:
display(spark.sql("""
SELECT 'batch_1' AS batch_name, COUNT(*) AS records
FROM batch_1_orders

UNION ALL

SELECT 'batch_2' AS batch_name, COUNT(*) AS records
FROM batch_2_orders

UNION ALL

SELECT 'batch_3' AS batch_name, COUNT(*) AS records
FROM batch_3_orders
"""))